In [1]:
import numpy as np

data = np.loadtxt('../data/titanic.csv', dtype=str, delimiter=',')

COLUMN_NAMES = data[0]
NUMPY_TITANIC = data[1:]

n_rows = NUMPY_TITANIC.shape[0]
print("Rows:", n_rows)
print("Columns:", NUMPY_TITANIC.shape[1])
print(NUMPY_TITANIC[:10])

for i in range(len(COLUMN_NAMES)):
    miss = np.sum(NUMPY_TITANIC[:, i] == '')
    pct = miss / n_rows * 100
    print(f"{COLUMN_NAMES[i]}: {miss} missing ({round(pct, 4)}%)")

Rows: 891
Columns: 15
[['0' '3' 'male' '22.0' '1' '0' '7.25' 'S' 'Third' 'man' 'True' ''
  'Southampton' 'no' 'False']
 ['1' '1' 'female' '38.0' '1' '0' '71.2833' 'C' 'First' 'woman' 'False'
  'C' 'Cherbourg' 'yes' 'False']
 ['1' '3' 'female' '26.0' '0' '0' '7.925' 'S' 'Third' 'woman' 'False' ''
  'Southampton' 'yes' 'True']
 ['1' '1' 'female' '35.0' '1' '0' '53.1' 'S' 'First' 'woman' 'False' 'C'
  'Southampton' 'yes' 'False']
 ['0' '3' 'male' '35.0' '0' '0' '8.05' 'S' 'Third' 'man' 'True' ''
  'Southampton' 'no' 'True']
 ['0' '3' 'male' '' '0' '0' '8.4583' 'Q' 'Third' 'man' 'True' ''
  'Queenstown' 'no' 'True']
 ['0' '1' 'male' '54.0' '0' '0' '51.8625' 'S' 'First' 'man' 'True' 'E'
  'Southampton' 'no' 'True']
 ['0' '3' 'male' '2.0' '3' '1' '21.075' 'S' 'Third' 'child' 'False' ''
  'Southampton' 'no' 'False']
 ['1' '3' 'female' '27.0' '0' '2' '11.1333' 'S' 'Third' 'woman' 'False'
  '' 'Southampton' 'yes' 'False']
 ['1' '2' 'female' '14.0' '1' '0' '30.0708' 'C' 'Second' 'child' 'False'


In [2]:
# class = pclass, embark_town = embarked, alive = survived (same info, diff labels)
redundant = ['class', 'embark_town', 'alive']

high_missing = []
for i in range(len(COLUMN_NAMES)):
    if np.sum(NUMPY_TITANIC[:, i] == '') > n_rows / 2:
        high_missing.append(str(COLUMN_NAMES[i]))

keep = []
for i in range(len(COLUMN_NAMES)):
    name = str(COLUMN_NAMES[i])
    if name not in redundant and name not in high_missing:
        keep.append(i)

NUMPY_TITANIC = NUMPY_TITANIC[:, keep]
COLUMN_NAMES = COLUMN_NAMES[keep]

print("Redundant columns removed:", redundant)
print("Columns with more than 50% missing removed:", high_missing)
print("Remaining columns:", COLUMN_NAMES.tolist())

Redundant columns removed: ['class', 'embark_town', 'alive']
Columns with more than 50% missing removed: ['deck']
Remaining columns: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'who', 'adult_male', 'alone']


In [3]:
target_col = 'survived'
num_cols = ['age', 'sibsp', 'parch', 'fare']
cat_cols = ['pclass', 'sex', 'embarked', 'who', 'adult_male', 'alone']

def col_idx(name):
    return np.where(COLUMN_NAMES == name)[0][0]

print("Target column:", target_col)
print("Numerical features:", num_cols)
print("Categorical features:", cat_cols)

Target column: survived
Numerical features: ['age', 'sibsp', 'parch', 'fare']
Categorical features: ['pclass', 'sex', 'embarked', 'who', 'adult_male', 'alone']


In [4]:
num_idx = [col_idx(c) for c in num_cols]

temp = NUMPY_TITANIC[:, num_idx].copy()
temp[temp == ''] = 'nan'
NUMERICAL_FEATURES = temp.astype(float)

miss_dict = {}
for j in range(len(num_cols)):
    miss_dict[num_cols[j]] = int(np.sum(np.isnan(NUMERICAL_FEATURES[:, j])))

print("Data type of NUMERICAL_FEATURES:", NUMERICAL_FEATURES.dtype)
print("Missing values per numerical feature:", miss_dict)

Data type of NUMERICAL_FEATURES: float64
Missing values per numerical feature: {'age': 177, 'sibsp': 0, 'parch': 0, 'fare': 0}


In [5]:
for c in cat_cols:
    vals, counts = np.unique(NUMPY_TITANIC[:, col_idx(c)], return_counts=True)
    d = {}
    for v, cnt in zip(vals, counts):
        d[str(v)] = int(cnt)
    print(f"{c}: {d}")

pclass: {'1': 216, '2': 184, '3': 491}
sex: {'female': 314, 'male': 577}
embarked: {'': 2, 'C': 168, 'Q': 77, 'S': 644}
who: {'child': 83, 'man': 537, 'woman': 271}
adult_male: {'False': 354, 'True': 537}
alone: {'False': 354, 'True': 537}


In [6]:
def get_mode(arr):
    vals, counts = np.unique(arr, return_counts=True)
    # unique is sorted, argmax picks first max -> smallest value on tie
    return vals[np.argmax(counts)]

def r4(arr):
    return [round(float(x), 4) for x in arr]

In [7]:
print("Without excluding missing elements:")
modes = []
for j in range(NUMERICAL_FEATURES.shape[1]):
    modes.append(get_mode(NUMERICAL_FEATURES[:, j]))

print("Mean:", r4(np.mean(NUMERICAL_FEATURES, axis=0)))
print("Median:", r4(np.median(NUMERICAL_FEATURES, axis=0)))
print("Mode:", r4(modes))
print("Std:", r4(np.std(NUMERICAL_FEATURES, axis=0)))
print("Min:", r4(np.min(NUMERICAL_FEATURES, axis=0)))
print("Max:", r4(np.max(NUMERICAL_FEATURES, axis=0)))

Without excluding missing elements:
Mean: [nan, 0.523, 0.3816, 32.2042]
Median: [nan, 0.0, 0.0, 14.4542]
Mode: [nan, 0.0, 0.0, 8.05]
Std: [nan, 1.1021, 0.8056, 49.6655]
Min: [nan, 0.0, 0.0, 0.0]
Max: [nan, 8.0, 6.0, 512.3292]


In [8]:
print("Excluding missing elements:")
modes2 = []
for j in range(NUMERICAL_FEATURES.shape[1]):
    col = NUMERICAL_FEATURES[:, j]
    modes2.append(get_mode(col[~np.isnan(col)]))

print("Mean:", r4(np.nanmean(NUMERICAL_FEATURES, axis=0)))
print("Median:", r4(np.nanmedian(NUMERICAL_FEATURES, axis=0)))
print("Mode:", r4(modes2))
print("Std:", r4(np.nanstd(NUMERICAL_FEATURES, axis=0)))
print("Min:", r4(np.nanmin(NUMERICAL_FEATURES, axis=0)))
print("Max:", r4(np.nanmax(NUMERICAL_FEATURES, axis=0)))

Excluding missing elements:
Mean: [29.6991, 0.523, 0.3816, 32.2042]
Median: [28.0, 0.0, 0.0, 14.4542]
Mode: [24.0, 0.0, 0.0, 8.05]
Std: [14.5163, 1.1021, 0.8056, 49.6655]
Min: [0.42, 0.0, 0.0, 0.0]
Max: [80.0, 8.0, 6.0, 512.3292]


In [9]:
target = NUMPY_TITANIC[:, col_idx(target_col)].astype(float)
print("Mean of target column:", round(float(np.mean(target)), 4))

Mean of target column: 0.3838


In [10]:
cat_idx = [col_idx(c) for c in cat_cols]
CATEGORICAL_FEATURES = NUMPY_TITANIC[:, cat_idx].copy()

for j in range(len(cat_cols)):
    col = CATEGORICAL_FEATURES[:, j]
    miss = np.sum(col == '')
    if miss > 0:
        vals, counts = np.unique(col[col != ''], return_counts=True)
        most = vals[np.argmax(counts)]
        CATEGORICAL_FEATURES[col == '', j] = most
        print(f"{cat_cols[j]}: {miss} missing replaced with {most}")

print("Missing values remaining in categorical features:", np.sum(CATEGORICAL_FEATURES == ''))

embarked: 2 missing replaced with S
Missing values remaining in categorical features: 0


In [11]:
ORDINAL_ENCODED = np.zeros(CATEGORICAL_FEATURES.shape, dtype=int)

for j in range(len(cat_cols)):
    cats, codes = np.unique(CATEGORICAL_FEATURES[:, j], return_inverse=True)
    ORDINAL_ENCODED[:, j] = codes
    mapping = {}
    for k in range(len(cats)):
        mapping[str(cats[k])] = k
    print(f"{cat_cols[j]}: {mapping}")

print("Ordinal encoded (first 5 rows):")
print(ORDINAL_ENCODED[:5])

pclass: {'1': 0, '2': 1, '3': 2}
sex: {'female': 0, 'male': 1}
embarked: {'C': 0, 'Q': 1, 'S': 2}
who: {'child': 0, 'man': 1, 'woman': 2}
adult_male: {'False': 0, 'True': 1}
alone: {'False': 0, 'True': 1}
Ordinal encoded (first 5 rows):
[[2 1 2 1 1 0]
 [0 0 0 2 0 0]
 [2 0 2 2 0 1]
 [0 0 2 2 0 0]
 [2 1 2 1 1 1]]


In [12]:
parts = []
for j in range(len(cat_cols)):
    col = CATEGORICAL_FEATURES[:, j]
    cats = np.unique(col)
    parts.append((col.reshape(-1, 1) == cats).astype(int))

ONEHOT_ENCODED = np.hstack(parts)

print("One-hot encoded (first 5 rows):")
print(ONEHOT_ENCODED[:5])

One-hot encoded (first 5 rows):
[[0 0 1 0 1 0 0 1 0 1 0 0 1 1 0]
 [1 0 0 1 0 1 0 0 0 0 1 1 0 1 0]
 [0 0 1 1 0 0 0 1 0 0 1 1 0 0 1]
 [1 0 0 1 0 0 0 1 0 0 1 1 0 1 0]
 [0 0 1 0 1 0 0 1 0 1 0 0 1 0 1]]


In [13]:
NUMERICAL_ORDINAL = np.hstack([NUMERICAL_FEATURES, ORDINAL_ENCODED])
NUMERICAL_ONEHOT = np.hstack([NUMERICAL_FEATURES, ONEHOT_ENCODED])

print("Shape of NUMERICAL_ORDINAL:", NUMERICAL_ORDINAL.shape)
print("Shape of NUMERICAL_ONEHOT:", NUMERICAL_ONEHOT.shape)

Shape of NUMERICAL_ORDINAL: (891, 10)
Shape of NUMERICAL_ONEHOT: (891, 19)


In [14]:
col_means = np.nanmean(NUMERICAL_FEATURES, axis=0)
MEAN_IMPUTED = NUMERICAL_FEATURES.copy()
for j in range(MEAN_IMPUTED.shape[1]):
    mask = np.isnan(MEAN_IMPUTED[:, j])
    MEAN_IMPUTED[mask, j] = col_means[j]
print("Mean imputation fill values:", r4(col_means))

col_medians = np.nanmedian(NUMERICAL_FEATURES, axis=0)
MEDIAN_IMPUTED = NUMERICAL_FEATURES.copy()
for j in range(MEDIAN_IMPUTED.shape[1]):
    mask = np.isnan(MEDIAN_IMPUTED[:, j])
    MEDIAN_IMPUTED[mask, j] = col_medians[j]
print("Median imputation fill values:", r4(col_medians))

Mean imputation fill values: [29.6991, 0.523, 0.3816, 32.2042]
Median imputation fill values: [28.0, 0.0, 0.0, 14.4542]


In [15]:
for j in range(len(num_cols)):
    if miss_dict[num_cols[j]] > 0:
        print(f"{num_cols[j]}:")
        print(f"NUMERICAL_FEATURES: mean {round(float(np.nanmean(NUMERICAL_FEATURES[:, j])), 4)}, std {round(float(np.nanstd(NUMERICAL_FEATURES[:, j])), 4)}")
        print(f"MEAN_IMPUTED: mean {round(float(np.mean(MEAN_IMPUTED[:, j])), 4)}, std {round(float(np.std(MEAN_IMPUTED[:, j])), 4)}")
        print(f"MEDIAN_IMPUTED: mean {round(float(np.mean(MEDIAN_IMPUTED[:, j])), 4)}, std {round(float(np.std(MEDIAN_IMPUTED[:, j])), 4)}")

age:
NUMERICAL_FEATURES: mean 29.6991, std 14.5163
MEAN_IMPUTED: mean 29.6991, std 12.9947
MEDIAN_IMPUTED: mean 29.3616, std 13.0124


In [16]:
mins = np.min(MEDIAN_IMPUTED, axis=0)
maxs = np.max(MEDIAN_IMPUTED, axis=0)
MINMAX_NORMALISED = (MEDIAN_IMPUTED - mins) / (maxs - mins)

print("MINMAX_NORMALISED (first 5 rows):")
print(np.round(MINMAX_NORMALISED[:5], 4))

MINMAX_NORMALISED (first 5 rows):
[[0.2712 0.125  0.     0.0142]
 [0.4722 0.125  0.     0.1391]
 [0.3214 0.     0.     0.0155]
 [0.4345 0.125  0.     0.1036]
 [0.4345 0.     0.     0.0157]]


In [17]:
mu = np.mean(MEDIAN_IMPUTED, axis=0)
sigma = np.std(MEDIAN_IMPUTED, axis=0)
ZSCORE_STANDARDISED = (MEDIAN_IMPUTED - mu) / sigma

print("ZSCORE_STANDARDISED (first 5 rows):")
print(np.round(ZSCORE_STANDARDISED[:5], 4))

ZSCORE_STANDARDISED (first 5 rows):
[[-0.5657  0.4328 -0.4737 -0.5024]
 [ 0.6639  0.4328 -0.4737  0.7868]
 [-0.2583 -0.4745 -0.4737 -0.4889]
 [ 0.4333  0.4328 -0.4737  0.4207]
 [ 0.4333 -0.4745 -0.4737 -0.4863]]
